# 第 23 课：AI 系统设计与部署

## 学习目标
- 理解一个 AI 模型从训练完成到上线服务的完整链路
- 掌握模型服务的核心架构模式：在线推理 / 异步推理 / 边缘部署
- 理解 MLOps 的核心概念：模型注册、版本管理、A/B 测试、监控告警
- 建立对 AI 系统可靠性、安全性和成本优化的工程直觉

## 在学习路线中的位置
前 22 课，我们从线性回归一路学到训练与推理工程。现在你已经知道**怎么训练一个模型**和**怎么让推理跑得快**。但一个残酷的现实是：能跑的模型 ≠ 能用的产品。

本课是「从懂模型到能建系统」的最后一跃——把训练好的模型变成一个可靠、可观测、可持续迭代的 AI 服务。

## 核心概念：为什么部署这么难？

传统软件部署：代码写好 → 打包 → 部署 → 如果有 bug 就修代码

AI 模型部署的额外挑战：
1. **数据漂移**：用户行为会变化，训练数据的分布和线上真实数据逐渐偏移
2. **模型退化**：没有 bug，但预测质量随时间下降
3. **延迟约束**：LLM 生成一个回答可能需要 2-10 秒，用户能等多久？
4. **成本控制**：GPU 很贵，如何在 SLA 和成本之间找到平衡？
5. **安全风险**：Prompt 注入、数据泄露、幻觉输出

这就是为什么 AI 系统需要一套完整的工程体系来支撑。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict
import time
import json

plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("环境准备完成 ✅")

## 1. 模型服务架构模式

三种核心部署模式：
- **在线推理（Online Inference）**：请求 → 即时返回结果。延迟敏感（< 100ms），如搜索排序、推荐
- **异步推理（Async Inference）**：请求 → 排队 → 结果回调。延迟容忍（秒级），如 LLM 生成、视频处理
- **边缘部署（Edge Deployment）**：模型部署在终端设备上，如手机、IoT 设备

直觉类比：
- 在线推理 = 快餐店（下单后几分钟内拿到）
- 异步推理 = 外卖（下单后等配送，可以干别的事）
- 边缘部署 = 自己做饭（食材和菜谱都在手边）

In [ ]:
# 可视化：三种服务架构模式的延迟-吞吐量权衡
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

modes = {
    '在线推理\n(Online)': {'latency': 50, 'throughput': 1000, 'color': '#C96442', 'x': 50, 'y': 1000},
    '异步推理\n(Async)': {'latency': 5000, 'throughput': 200, 'color': '#4A90D9', 'x': 5000, 'y': 200},
    '边缘部署\n(Edge)': {'latency': 20, 'throughput': 10, 'color': '#5B8C5A', 'x': 20, 'y': 10},
}

for name, props in modes.items():
    ax.scatter(props['x'], props['y'], s=300, c=props['color'], zorder=5, edgecolors='white', linewidth=2)
    ax.annotate(name, (props['x'], props['y']), textcoords='offset points', 
                xytext=(15, 10), fontsize=11, fontweight='bold')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('典型延迟 (ms)', fontsize=12)
ax.set_ylabel('吞吐量 (req/s)', fontsize=12)
ax.set_title('AI 模型服务架构模式对比', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

# 添加象限标注
ax.axhline(y=100, color='gray', linestyle='--', alpha=0.3)
ax.axvline(x=100, color='gray', linestyle='--', alpha=0.3)
ax.text(5, 2000, '低延迟 + 高吞吐\n= 在线服务理想区', fontsize=9, color='gray', alpha=0.6)
ax.text(2000, 2000, '高延迟 + 高吞吐\n= 批处理理想区', fontsize=9, color='gray', alpha=0.6)

plt.tight_layout()
plt.savefig('docs/assets/lesson23-serving-modes.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ 服务架构模式图生成完成")

## 2. 模型版本管理与 A/B 测试

为什么需要版本管理？
- 模型会持续迭代（v1 → v2 → v3）
- 不同模型在不同场景下表现不同
- 出问题时需要快速回滚

A/B 测试流程：
1. 流量分流：90% → 模型 A（当前版本），10% → 模型 B（新版本）
2. 收集指标：准确率、延迟、用户满意度、转化率
3. 统计检验：确认 B 是否显著优于 A
4. 决策：全量切换 B，或继续调优，或回滚

直觉：就像药物临床试验——新药先给少部分人试，有效且安全后再大规模推广。

In [ ]:
# 模拟 A/B 测试：两个模型版本的性能对比
np.random.seed(42)

# 模型 A：基线模型（准确率 85%）
model_a_correct = np.random.binomial(1, 0.85, 1000)
# 模型 B：新版模型（准确率 88%）
model_b_correct = np.random.binomial(1, 0.88, 1000)

# 模拟随时间累积的准确率
cumulative_a = np.cumsum(model_a_correct) / np.arange(1, 1001)
cumulative_b = np.cumsum(model_b_correct) / np.arange(1, 1001)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 左图：累积准确率对比
ax1.plot(cumulative_a, color='#C96442', alpha=0.8, label='模型 A (baseline)')
ax1.plot(cumulative_b, color='#4A90D9', alpha=0.8, label='模型 B (new)')
ax1.axhline(y=0.85, color='#C96442', linestyle='--', alpha=0.4)
ax1.axhline(y=0.88, color='#4A90D9', linestyle='--', alpha=0.4)
ax1.set_xlabel('请求数', fontsize=11)
ax1.set_ylabel('累积准确率', fontsize=11)
ax1.set_title('A/B 测试：累积准确率对比', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.set_ylim(0.78, 0.95)
ax1.grid(True, alpha=0.3)

# 右图：延迟分布对比
latency_a = np.random.lognormal(mean=3.5, sigma=0.5, size=1000)  # 基线延迟
latency_b = np.random.lognormal(mean=3.3, sigma=0.4, size=1000)  # 新模型稍快

ax2.hist(latency_a, bins=40, alpha=0.6, color='#C96442', label=f'模型 A (P50={np.median(latency_a):.0f}ms)')
ax2.hist(latency_b, bins=40, alpha=0.6, color='#4A90D9', label=f'模型 B (P50={np.median(latency_b):.0f}ms)')
ax2.set_xlabel('延迟 (ms)', fontsize=11)
ax2.set_ylabel('请求数', fontsize=11)
ax2.set_title('A/B 测试：延迟分布对比', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('docs/assets/lesson23-ab-test.png', dpi=150, bbox_inches='tight')
plt.show()

# 统计显著性检验
from scipy import stats
t_stat, p_value = stats.ttest_ind(model_b_correct, model_a_correct)
print(f"\n📊 A/B 测试统计结果:")
print(f"  模型 A 准确率: {model_a_correct.mean():.4f}")
print(f"  模型 B 准确率: {model_b_correct.mean():.4f}")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {p_value:.4f}")
print(f"  结论: {'模型 B 显著优于 A ✅' if p_value < 0.05 else '差异不显著，需要更多数据 ⚠️'}")

## 3. 数据漂移检测

数据漂移（Data Drift）是 AI 系统最大的「隐形杀手」：模型没有 bug，但预测质量持续下降。

常见漂移类型：
- **协变量漂移**：输入特征的分布变了（用户群体变化）
- **概念漂移**：输入和输出的关系变了（疫情前后的消费行为完全不同）
- **标签漂移**：输出的分布变了（正负样本比例变化）

检测方法：KL 散度、PSI（Population Stability Index）、KS 检验

In [ ]:
# 模拟数据漂移检测
np.random.seed(42)

# 训练时数据分布
train_dist = np.random.normal(50, 10, 5000)

# 模拟 12 个月的数据分布逐渐漂移
months = ['1月', '2月', '3月', '4月', '5月', '6月', 
          '7月', '8月', '9月', '10月', '11月', '12月']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 左图：各月数据分布变化
drift_amounts = np.linspace(0, 15, 12)
colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, 12))

for i, (month, drift) in enumerate(zip(months, drift_amounts)):
    month_dist = np.random.normal(50 + drift, 10 + drift*0.2, 1000)
    ax1.hist(month_dist, bins=30, alpha=0.3, color=colors[i], label=month if i % 3 == 0 else '')

ax1.axvline(x=50, color='#C96442', linestyle='--', linewidth=2, label='训练数据均值')
ax1.set_xlabel('特征值', fontsize=11)
ax1.set_ylabel('频次', fontsize=11)
ax1.set_title('数据漂移：月度分布变化', fontsize=13, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# 右图：PSI 指标随时间变化
def calculate_psi(expected, actual, buckets=10):
    """计算 Population Stability Index"""
    breakpoints = np.linspace(min(expected.min(), actual.min()), 
                             max(expected.max(), actual.max()), buckets + 1)
    expected_pct = np.histogram(expected, bins=breakpoints)[0] / len(expected)
    actual_pct = np.histogram(actual, bins=breakpoints)[0] / len(actual)
    # 避免除零
    expected_pct = np.clip(expected_pct, 0.0001, None)
    actual_pct = np.clip(actual_pct, 0.0001, None)
    psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))
    return psi

psi_values = []
accuracy_values = []
for drift in drift_amounts:
    month_data = np.random.normal(50 + drift, 10 + drift*0.2, 2000)
    psi = calculate_psi(train_dist, month_data)
    psi_values.append(psi)
    # 模拟准确率随漂移下降
    acc = max(0.5, 0.95 - drift * 0.03)
    accuracy_values.append(acc)

ax2_twin = ax2.twinx()
ax2.bar(months, psi_values, color=['#5B8C5A' if p < 0.1 else '#E8A838' if p < 0.25 else '#C96442' for p in psi_values], alpha=0.7, label='PSI')
ax2_twin.plot(months, accuracy_values, 'o-', color='#4A90D9', linewidth=2, markersize=6, label='准确率')

ax2.axhline(y=0.1, color='#E8A838', linestyle='--', alpha=0.5, label='PSI 警戒线 (0.1)')
ax2.axhline(y=0.25, color='#C96442', linestyle='--', alpha=0.5, label='PSI 危险线 (0.25)')

ax2.set_xlabel('月份', fontsize=11)
ax2.set_ylabel('PSI 值', fontsize=11, color='#C96442')
ax2_twin.set_ylabel('模型准确率', fontsize=11, color='#4A90D9')
ax2.set_title('数据漂移指标 (PSI) 与模型准确率', fontsize=13, fontweight='bold')

# 合并图例
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='center right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('docs/assets/lesson23-data-drift.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 PSI 指标解读:")
print("  PSI < 0.1  → 分布稳定 ✅")
print("  PSI 0.1~0.25 → 轻微漂移，需关注 ⚠️")
print("  PSI > 0.25 → 严重漂移，需重新训练 🔴")

## 4. AI 系统全链路架构

一个完整的 AI 系统不只是「模型 + API」，而是包含多层组件的工程体系：

```
┌─────────────────────────────────────────────────────┐
│                    客户端 (Web/App/SDK)              │
├─────────────────────────────────────────────────────┤
│              API 网关 (认证、限流、路由)              │
├──────────┬──────────┬───────────┬───────────────────┤
│ 模型服务  │ 向量检索  │ 特征存储  │  Prompt 管理      │
│ (推理引擎)│ (Milvus)  │ (Redis)  │  (模板/版本)      │
├──────────┴──────────┴───────────┴───────────────────┤
│             监控层 (延迟/准确率/漂移/成本)            │
├─────────────────────────────────────────────────────┤
│            数据管线 (标注/清洗/特征工程)              │
├─────────────────────────────────────────────────────┤
│            训练管线 (实验管理/模型注册)               │
└─────────────────────────────────────────────────────┘
```

关键工程决策点：
- **推理引擎选型**：vLLM / Triton / TGI / 自研
- **向量数据库**：Milvus / Pinecone / Weaviate / pgvector
- **特征存储**：Feast / Tecton / Redis + 自研
- **实验管理**：MLflow / Weights & Biases / 自研
- **部署方式**：Docker + K8s / Serverless / 边缘

In [ ]:
# 模拟 AI 系统关键指标的 SLA 监控面板
np.random.seed(42)

hours = np.arange(0, 24)

# 模拟一天的监控数据
latency_p50 = np.random.lognormal(3.0, 0.2, 24)  # P50 延迟
latency_p99 = latency_p50 * np.random.uniform(3, 6, 24)  # P99 延迟
error_rate = np.random.exponential(0.002, 24)  # 错误率
qps = np.concatenate([
    np.random.poisson(50, 6),    # 凌晨低谷
    np.random.poisson(200, 6),   # 上午高峰
    np.random.poisson(300, 6),   # 下午高峰
    np.random.poisson(100, 6),   # 晚间
])
gpu_util = np.clip(0.3 + qps / 600 + np.random.normal(0, 0.05, 24), 0, 1)

# 模拟一次异常事件
latency_p50[14] *= 3  # 下午 2 点延迟突增
latency_p99[14] *= 4
error_rate[14] = 0.05  # 错误率飙升

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 请求量
axes[0,0].fill_between(hours, qps, alpha=0.3, color='#4A90D9')
axes[0,0].plot(hours, qps, color='#4A90D9', linewidth=2)
axes[0,0].set_title('请求量 (QPS)', fontsize=12, fontweight='bold')
axes[0,0].set_ylabel('Requests/s')
axes[0,0].grid(True, alpha=0.3)

# 2. 延迟
axes[0,1].semilogy(hours, latency_p50, 'o-', color='#5B8C5A', label='P50', markersize=4)
axes[0,1].semilogy(hours, latency_p99, 's-', color='#C96442', label='P99', markersize=4)
axes[0,1].axhline(y=500, color='red', linestyle='--', alpha=0.5, label='SLA (500ms)')
axes[0,1].axvspan(13.5, 14.5, alpha=0.2, color='red', label='异常时段')
axes[0,1].set_title('推理延迟', fontsize=12, fontweight='bold')
axes[0,1].set_ylabel('延迟 (ms)')
axes[0,1].legend(fontsize=9)
axes[0,1].grid(True, alpha=0.3)

# 3. 错误率
axes[1,0].bar(hours, error_rate * 100, 
              color=['#C96442' if e > 0.01 else '#5B8C5A' for e in error_rate], alpha=0.7)
axes[1,0].axhline(y=1, color='red', linestyle='--', alpha=0.5, label='SLA (1%)')
axes[1,0].set_title('错误率', fontsize=12, fontweight='bold')
axes[1,0].set_ylabel('错误率 (%)')
axes[1,0].set_xlabel('小时')
axes[1,0].legend(fontsize=9)
axes[1,0].grid(True, alpha=0.3)

# 4. GPU 利用率
axes[1,1].fill_between(hours, gpu_util * 100, alpha=0.3, color='#E8A838')
axes[1,1].plot(hours, gpu_util * 100, color='#E8A838', linewidth=2)
axes[1,1].axhline(y=80, color='red', linestyle='--', alpha=0.5, label='高负载警戒 (80%)')
axes[1,1].set_title('GPU 利用率', fontsize=12, fontweight='bold')
axes[1,1].set_ylabel('利用率 (%)')
axes[1,1].set_xlabel('小时')
axes[1,1].legend(fontsize=9)
axes[1,1].grid(True, alpha=0.3)

plt.suptitle('AI 系统监控面板 — 24 小时指标概览', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('docs/assets/lesson23-monitoring.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 监控指标解读:")
print("  14:00 左右出现明显异常 — 延迟飙升、错误率上升")
print("  典型原因: 流量突增 / GPU 显存不足 / 上游服务超时")
print("  应对策略: 自动弹性扩容 → 告警通知 → 故障复盘")

## 5. 成本优化策略

AI 系统的运营成本主要来自 GPU 计算，关键优化维度：

| 策略 | 效果 | 适用场景 |
|------|------|----------|
| 量化 (INT4/INT8) | 成本降 2-4x | 推理场景 |
| 动态批处理 | 吞吐量提升 3-5x | 在线推理 |
| 弹性伸缩 | 节省 30-60% 空闲成本 | 流量波动大 |
| 模型蒸馏 | 推理成本降 10x+ | 大模型部署 |
| Speculative Decoding | 延迟降 2-3x | 自回归生成 |
| 缓存 (语义缓存) | 重复查询成本降 90%+ | RAG/对话场景 |

In [ ]:
# 模拟不同部署方案的成本-性能权衡
strategies = {
    'FP16 原始\n(基线)': {'cost': 4.0, 'qps': 20, 'latency': 100, 'accuracy': 100.0},
    'INT8 量化': {'cost': 2.0, 'qps': 35, 'latency': 70, 'accuracy': 99.5},
    'INT4 量化': {'cost': 1.0, 'qps': 50, 'latency': 50, 'accuracy': 98.0},
    '蒸馏小模型': {'cost': 0.3, 'qps': 120, 'latency': 20, 'accuracy': 94.0},
    'FP16 + 动态批': {'cost': 4.0, 'qps': 60, 'latency': 200, 'accuracy': 100.0},
    'INT4 + 缓存': {'cost': 0.5, 'qps': 100, 'latency': 30, 'accuracy': 98.0},
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

names = list(strategies.keys())
costs = [v['cost'] for v in strategies.values()]
accuracies = [v['accuracy'] for v in strategies.values()]
latencies = [v['latency'] for v in strategies.values()]
qpss = [v['qps'] for v in strategies.values()]

colors = ['#C96442', '#4A90D9', '#5B8C5A', '#E8A838', '#9B59B6', '#2ECC71']

# 左图：成本 vs 准确率
for i, name in enumerate(names):
    ax1.scatter(costs[i], accuracies[i], s=200, c=colors[i], zorder=5, edgecolors='white', linewidth=2)
    offset = (15, 5) if '蒸馏' not in name else (15, -15)
    ax1.annotate(name.replace('\n', ' '), (costs[i], accuracies[i]), 
                textcoords='offset points', xytext=offset, fontsize=9)

ax1.set_xlabel('GPU 成本 (相对基线)', fontsize=11)
ax1.set_ylabel('相对准确率 (%)', fontsize=11)
ax1.set_title('部署策略：成本 vs 准确率', fontsize=13, fontweight='bold')
ax1.set_ylim(92, 101)
ax1.grid(True, alpha=0.3)

# 右图：延迟 vs 吞吐量
for i, name in enumerate(names):
    ax2.scatter(latencies[i], qpss[i], s=[c*80 for c in costs[i] for c in [costs[i]]], 
               c=colors[i], zorder=5, edgecolors='white', linewidth=2, alpha=0.8)
    ax2.annotate(name.replace('\n', ' '), (latencies[i], qpss[i]), 
                textcoords='offset points', xytext=(10, 5), fontsize=9)

ax2.set_xlabel('延迟 (ms)', fontsize=11)
ax2.set_ylabel('吞吐量 (req/s)', fontsize=11)
ax2.set_title('部署策略：延迟 vs 吞吐量', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('docs/assets/lesson23-cost-tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n💰 成本优化核心结论:")
print("  1. INT4 量化是性价比之王 — 成本降 4x，准确率仅损失 2%")
print("  2. 蒸馏小模型适合对延迟极敏感的场景 — 但准确率折损更大")
print("  3. 语义缓存对重复查询场景效果惊人 — 命中率 70%+ 即可大幅降本")
print("  4. 没有银弹，核心是理解你的业务 SLA 约束后选择最优组合")

## 6. AI 安全与防护

AI 系统上线后面临的安全风险：

| 风险 | 描述 | 防护措施 |
|------|------|----------|
| Prompt 注入 | 用户通过精心构造的输入操控模型行为 | 输入过滤 / 系统提示隔离 |
| 数据泄露 | 模型在输出中暴露训练数据中的敏感信息 | 差分隐私 / 输出过滤 |
| 幻觉输出 | 模型生成看似合理但完全错误的内容 | RAG 接地 / 事实性校验 |
| 对抗样本 | 微小扰动输入导致模型严重误判 | 对抗训练 / 输入校验 |
| 供应链攻击 | 恶意模型/数据注入系统 | 模型签名 / 可信来源 |

架构师视角：安全不是事后补丁，而是架构设计的一等公民。

In [ ]:
# 模拟一个最小化的 AI 服务监控仪表板
class AIServiceMonitor:
    """模拟 AI 服务的监控指标收集"""
    
    def __init__(self, model_version='v2.3.1'):
        self.model_version = model_version
        self.metrics = defaultdict(list)
        self.alerts = []
        
    def log_request(self, latency_ms, success, tokens=0, cached=False):
        """记录一次请求"""
        self.metrics['latency'].append(latency_ms)
        self.metrics['success'].append(success)
        self.metrics['tokens'].append(tokens)
        self.metrics['cached'].append(cached)
        
        # 告警规则
        if latency_ms > 5000:
            self.alerts.append(f'🔴 延迟过高: {latency_ms:.0f}ms (SLA: 5000ms)')
        if not success:
            self.alerts.append(f'🔴 请求失败')
            
    def get_summary(self):
        """获取监控摘要"""
        if not self.metrics['latency']:
            return "暂无数据"
            
        latencies = np.array(self.metrics['latency'])
        successes = np.array(self.metrics['success'])
        tokens = np.array(self.metrics['tokens'])
        cached = np.array(self.metrics['cached'])
        
        total = len(latencies)
        cache_hit_rate = cached.mean() if len(cached) > 0 else 0
        
        summary = {
            'model_version': self.model_version,
            'total_requests': total,
            'success_rate': successes.mean(),
            'p50_latency': np.percentile(latencies, 50),
            'p99_latency': np.percentile(latencies, 99),
            'avg_tokens': tokens.mean(),
            'cache_hit_rate': cache_hit_rate,
            'alerts': self.alerts[-5:],  # 最近 5 条告警
            'status': '🟢 正常' if successes.mean() > 0.99 and np.percentile(latencies, 99) < 5000 else '🔴 异常'
        }
        return summary

# 模拟运行
monitor = AIServiceMonitor(model_version='v2.3.1')

np.random.seed(42)
for _ in range(1000):
    latency = np.random.lognormal(5.0, 0.5)
    success = np.random.random() > 0.005  # 99.5% 成功率
    tokens = np.random.randint(50, 500)
    cached = np.random.random() < 0.35  # 35% 缓存命中率
    monitor.log_request(latency, success, tokens, cached)

# 注入一次异常
monitor.log_request(8000, False, 0, False)

summary = monitor.get_summary()
print("=" * 50)
print("🧠 AI 服务监控仪表板")
print("=" * 50)
print(f"模型版本:    {summary['model_version']}")
print(f"总请求数:    {summary['total_requests']}")
print(f"成功率:      {summary['success_rate']:.4f} ({summary['success_rate']*100:.2f}%)")
print(f"P50 延迟:    {summary['p50_latency']:.0f}ms")
print(f"P99 延迟:    {summary['p99_latency']:.0f}ms")
print(f"平均 Token:  {summary['avg_tokens']:.0f}")
print(f"缓存命中率:  {summary['cache_hit_rate']:.2%}")
print(f"系统状态:    {summary['status']}")
print(f"\n最近告警:")
for alert in summary['alerts']:
    print(f"  {alert}")

## 总结

### 本课核心要点
1. **AI 系统不只是模型 + API**：还需要监控、版本管理、A/B 测试、漂移检测、安全防护
2. **三种服务模式各有适用场景**：在线推理（低延迟）、异步推理（高吞吐）、边缘部署（隐私/离线）
3. **数据漂移是隐形杀手**：用 PSI 等指标持续监控，PSI > 0.25 就要重新训练
4. **A/B 测试是模型迭代的标准方法**：先小流量验证，统计显著后再全量切换
5. **成本优化是工程问题**：量化、缓存、蒸馏、弹性伸缩——组合使用效果最好
6. **安全是架构的一等公民**：Prompt 注入、数据泄露、幻觉——都要在设计阶段考虑

### 课后思考
1. 如果你要部署一个 LLM 聊天机器人，会选择哪种服务架构？为什么？
2. 假设你的推荐系统准确率在过去 3 个月持续下降，你会怎么排查？
3. 一个模型的推理延迟从 200ms 涨到 2s，但准确率没变——这是不是一个问题？

### 关键代表项目与工具
- **MLOps 平台**：MLflow, Weights & Biases, Kubeflow
- **模型服务**：vLLM, NVIDIA Triton, HuggingFace TGI, TorchServe
- **向量数据库**：Milvus, Pinecone, Weaviate
- **特征存储**：Feast, Tecton
- **监控**：Prometheus + Grafana, Evidently AI
- **代表论文**："Hidden Technical Debt in Machine Learning Systems" (Sculley et al., 2015) —— 机器学习系统中隐藏的技术债